In [6]:
!pip -q install \
  pymupdf \
  groq \
  python-dotenv \
  httpx \
  feedparser \
  pandas \
  openpyxl \
  tqdm \
  gspread \
  google-auth \
  google-auth-oauthlib \
  google-auth-httplib2 \
  google-api-python-client
!pip -q install openai pandas numpy scikit-learn tqdm

# Cleaning the Dataset

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

##Loading the Data
df = pd.read_excel("/content/milestone2_final_200_labeled_groundtruth.xlsx")
print("Original shape:", df.shape)


Original shape: (200, 19)


In [12]:
columns_to_keep = [
    # Identifiers
    "subject_id",
    "hadm_id",

    # Demographics
    "gender",
    "admit_age",
    "race",

    # Primary LLM input
    "text",
    "radiology_notes",

    # AD/ADRD medications
    "donepezil",
    "memantine",
    "tacrine",
    "rivastigmine",
    "galantamine",

    # Cardiovascular medications
    "Enalapril",
    "Furosemide",
    "Lisinopril",
    "Metoprolol",
    "Spironolactone",

    # Ground truth
    "ground_truth"
]


In [13]:
df_clean = df[columns_to_keep].copy()

In [14]:
# ── STEP 5: Fill medication missing values ──
med_columns = [
    "donepezil", "memantine", "tacrine",
    "rivastigmine", "galantamine",
    "Enalapril", "Furosemide", "Lisinopril",
    "Metoprolol", "Spironolactone"
]
df_clean[med_columns] = df_clean[med_columns].fillna(0)

print("\nMedication nulls after fill:")
print(df_clean[med_columns].isnull().sum())


Medication nulls after fill:
donepezil         0
memantine         0
tacrine           0
rivastigmine      0
galantamine       0
Enalapril         0
Furosemide        0
Lisinopril        0
Metoprolol        0
Spironolactone    0
dtype: int64


In [15]:
missing_text = df_clean["text"].isnull().sum()
missing_radiology = df_clean["radiology_notes"].isnull().sum()

print(f"\nMissing discharge summaries: {missing_text}")
print(f"Missing radiology notes: {missing_radiology}")



Missing discharge summaries: 0
Missing radiology notes: 0


In [16]:
df_clean["radiology_notes"] = df_clean[
    "radiology_notes"
].fillna("No radiology notes available")

if missing_text > 0:
    print(f"Dropping {missing_text} patients — no discharge summary")
    df_clean = df_clean.dropna(subset=["text"])


In [17]:
print("\nGround truth distribution:")
print(df_clean["ground_truth"].value_counts())
print(f"Missing: {df_clean['ground_truth'].isnull().sum()}")


Ground truth distribution:
ground_truth
1    150
0     50
Name: count, dtype: int64
Missing: 0


In [19]:
df_clean.to_csv("milestone2_200_cleaned.csv", index=False)

In [18]:
import os
import ast
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [20]:
csv_path = "/content/milestone2_200_cleaned.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nGround truth distribution:")
print(df["ground_truth"].value_counts(dropna=False))

Shape: (200, 18)

Columns:
['subject_id', 'hadm_id', 'gender', 'admit_age', 'race', 'text', 'radiology_notes', 'donepezil', 'memantine', 'tacrine', 'rivastigmine', 'galantamine', 'Enalapril', 'Furosemide', 'Lisinopril', 'Metoprolol', 'Spironolactone', 'ground_truth']

Ground truth distribution:
ground_truth
1    150
0     50
Name: count, dtype: int64


In [21]:
data = df.copy()

data["ground_truth"] = pd.to_numeric(data["ground_truth"], errors="coerce")
data = data[data["ground_truth"].isin([0, 1])].copy()
data["ground_truth"] = data["ground_truth"].astype(int)

data["text"] = data["text"].fillna("").astype(str)
data["radiology_notes"] = data["radiology_notes"].fillna("").astype(str)

def build_combined_note(row):
    note_text = row["text"].strip()
    rad_text = row["radiology_notes"].strip()

    combined = f"""CLINICAL NOTE:
{note_text}

RADIOLOGY NOTES:
{rad_text}"""
    return combined.strip()

data["combined_note"] = data.apply(build_combined_note, axis=1)

MAX_CHARS = 12000
data["combined_note_trimmed"] = data["combined_note"].str.slice(0, MAX_CHARS)

print("Final usable rows:", len(data))
print("\nExample combined note preview:\n")
print(data.loc[0, "combined_note_trimmed"][:2000])

Final usable rows: 200

Example combined note preview:

CLINICAL NOTE:
Name:  ___               Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: NEUROLOGY
 
Allergies: 
Moxifloxacin / clindamycin / Bactrim
 
Attending: ___
 
Chief Complaint:
Seizure
 
Major Surgical or Invasive Procedure:
None
 
History of Present Illness:
The patient is an ___ right-handed woman with a history
of a fib on coumadin, HTN, diastolic CHF, severe TR, and mild
cognitive impairment with associated aphasia who presents as a
transfer from ___ after a focal motor seizure this
morning. Her daughter reports that she awoke feeling fine this
morning and seemed to be her usual self. They were talking 
around
9am when suddenly her left arm started to shake. She imitates it
as her shoulder and upper arm shaking up and down. There was no
shaking of any other part of her body. She remained awake and
alert throughout the episode and answered qu

In [22]:
train_df, test_df = train_test_split(
    data,
    test_size=80,
    stratify=data["ground_truth"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

print("\nTrain label distribution:")
print(train_df["ground_truth"].value_counts())

print("\nTest label distribution:")
print(test_df["ground_truth"].value_counts())

Train size: 120
Test size: 80

Train label distribution:
ground_truth
1    90
0    30
Name: count, dtype: int64

Test label distribution:
ground_truth
1    60
0    20
Name: count, dtype: int64


In [23]:
train_df.to_csv("milestone2_train_split.csv", index=False)
test_df.to_csv("milestone2_test_split.csv", index=False)

print("Saved milestone2_train_split.csv and milestone2_test_split.csv")

Saved milestone2_train_split.csv and milestone2_test_split.csv


# Import OpenAI

In [24]:
!pip -q install -U openai

import json
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

print("Client initialized:", api_key is not None and len(api_key) > 20)

Client initialized: True


In [25]:
from openai import OpenAI
from google.colab import userdata

api_key_raw = userdata.get("OPENAI_API_KEY")
# Filter out any non-ASCII characters to ensure a clean API key
api_key = "".join(char for char in api_key_raw if ord(char) < 128).strip()
client = OpenAI(api_key=api_key)

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with the single word OK"}],
    temperature=0
)

print(resp.choices[0].message.content)

OK


In [26]:
import json
import pandas as pd
from tqdm.auto import tqdm

In [27]:
def safe_val(x):
    if pd.isna(x):
        return "Not Tested"
    return str(x)

In [28]:
EXTRACTOR_SCHEMA = {
    "name": "adrd_extractor_output",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "subject_id": {"type": "string"},
            "age": {"type": "string"},
            "gender": {"type": "string"},
            "race": {"type": "string"},
            "cognitive_medications": {
                "type": "object",
                "properties": {
                    "ad_medications_present": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "cardiovascular_medications_present": {
                        "type": "array",
                        "items": {"type": "string"}
                    }
                },
                "required": [
                    "ad_medications_present",
                    "cardiovascular_medications_present"
                ],
                "additionalProperties": False
            },
            "lab_values": {
                "type": "object",
                "properties": {
                    "hemoglobin": {"type": "array", "items": {"type": "string"}},
                    "sodium": {"type": "array", "items": {"type": "string"}},
                    "potassium": {"type": "array", "items": {"type": "string"}},
                    "creatinine_serum": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["hemoglobin", "sodium", "potassium", "creatinine_serum"],
                "additionalProperties": False
            },
            "lab_trends": {
                "type": "object",
                "properties": {
                    "hemoglobin": {"type": "string"},
                    "sodium": {"type": "string"},
                    "potassium": {"type": "string"},
                    "creatinine_serum": {"type": "string"}
                },
                "required": ["hemoglobin", "sodium", "potassium", "creatinine_serum"],
                "additionalProperties": False
            },
            "cognitive_symptoms": {"type": "array", "items": {"type": "string"}},
            "radiology_findings": {"type": "array", "items": {"type": "string"}},
            "provider_findings": {"type": "array", "items": {"type": "string"}},
            "patient_complaints": {"type": "array", "items": {"type": "string"}},
            "temporal_references": {"type": "array", "items": {"type": "string"}},
            "other_relevant_findings": {"type": "array", "items": {"type": "string"}}
        },
        "required": [
            "subject_id",
            "age",
            "gender",
            "race",
            "cognitive_medications",
            "lab_values",
            "lab_trends",
            "cognitive_symptoms",
            "radiology_findings",
            "provider_findings",
            "patient_complaints",
            "temporal_references",
            "other_relevant_findings"
        ],
        "additionalProperties": False
    }
}

In [29]:
CLASSIFIER_SCHEMA = {
    "name": "adrd_classifier_output",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "subject_id": {"type": "string"},
            "classification": {
                "type": "string",
                "enum": ["YES", "NO", "UNCERTAIN"]
            },
            "confidence": {
                "type": "string",
                "enum": ["HIGH", "MEDIUM", "LOW"]
            },
            "strong_indicators": {
                "type": "array",
                "items": {"type": "string"}
            },
            "supporting_indicators": {
                "type": "array",
                "items": {"type": "string"}
            },
            "contradicting_indicators": {
                "type": "array",
                "items": {"type": "string"}
            },
            "reasoning": {"type": "string"},
            "conclusion": {"type": "string"}
        },
        "required": [
            "subject_id",
            "classification",
            "confidence",
            "strong_indicators",
            "supporting_indicators",
            "contradicting_indicators",
            "reasoning",
            "conclusion"
        ],
        "additionalProperties": False
    }
}

In [30]:
print(df.columns.tolist())

['subject_id', 'hadm_id', 'gender', 'admit_age', 'race', 'text', 'radiology_notes', 'donepezil', 'memantine', 'tacrine', 'rivastigmine', 'galantamine', 'Enalapril', 'Furosemide', 'Lisinopril', 'Metoprolol', 'Spironolactone', 'ground_truth']


In [31]:
def build_extractor_prompt(row):
    return f"""
You are a senior neurologist and clinical informatics expert specializing in Alzheimer's Disease and Related Dementias (AD/ADRD). You are part of a clinical team tasked with determining whether a patient has been diagnosed with AD/ADRD.

Your specific task is to extract relevant clinical information from the patient's chart that will be used in a subsequent classification step.

═══════════════════════════════
PATIENT INFORMATION
═══════════════════════════════
Patient ID:   {safe_val(row.get('subject_id'))}
Age:          {safe_val(row.get('admit_age'))}
Gender:       {safe_val(row.get('gender'))}
Race:         {safe_val(row.get('race_group'))}

─── Discharge Summary ───
{safe_val(row.get('text'))}

─── Radiology Notes ───
{safe_val(row.get('radiology_notes'))}

─── AD/ADRD Medications ───
(0 = not administered, 1 = administered during encounter)
Donepezil:    {safe_val(row.get('donepezil'))}
Memantine:    {safe_val(row.get('memantine'))}
Tacrine:      {safe_val(row.get('tacrine'))}
Rivastigmine: {safe_val(row.get('rivastigmine'))}
Galantamine:  {safe_val(row.get('galantamine'))}

─── Cardiovascular Medications ───
(0 = not administered, 1 = administered during encounter)
Enalapril:       {safe_val(row.get('Enalapril'))}
Furosemide:      {safe_val(row.get('Furosemide'))}
Lisinopril:      {safe_val(row.get('Lisinopril'))}
Metoprolol:      {safe_val(row.get('Metoprolol'))}
Spironolactone:  {safe_val(row.get('Spironolactone'))}

─── Laboratory Values ───
(Multiple values represent longitudinal readings, earliest to most recent)
(Not Tested = lab was not collected during this encounter)
Hemoglobin (normal 12-17 g/dL):          {safe_val(row.get('hemoglobin'))}
Sodium (normal 135-145 mEq/L):           {safe_val(row.get('sodium'))}
Potassium (normal 3.5-5.0 mEq/L):        {safe_val(row.get('potassium'))}
Creatinine Serum (normal 0.6-1.2 mg/dL): {safe_val(row.get('creatinine'))}

Extract the following clinical information.
When extracting, pay special attention to:
- Any mention of memory loss, confusion, or cognitive decline
- References to prior dementia or MCI diagnosis
- Temporal references to symptom history
- Cognitive test scores (MMSE, MoCA, etc.)
- Brain imaging findings suggesting neurodegeneration
- AD medications as strong diagnostic signals
- Cardiovascular conditions as known AD/ADRD risk factors
- Abnormal or trending lab values

If information is not available or not relevant,
write "None identified" for that field.
""".strip()

In [32]:
def build_classifier_prompt(row, extracted_json_str):
    return f"""
You are a senior neurological diagnostician specializing in Alzheimer's Disease and Related Dementias (AD/ADRD). You are presenting your diagnostic findings to a panel of clinical peers.

Your task is to examine the extracted clinical information and original discharge summary below and determine whether this patient has AD/ADRD.

═══════════════════════════════
EXTRACTED CLINICAL INFORMATION
═══════════════════════════════
{extracted_json_str}

═══════════════════════════════
ORIGINAL DISCHARGE SUMMARY
═══════════════════════════════
{safe_val(row.get('text'))}

═══════════════════════════════
CLASSIFICATION INSTRUCTIONS
═══════════════════════════════
Using the clinical evidence above, classify this patient as YES, NO, or UNCERTAIN for AD/ADRD.

Use the following clinical framework to guide your decision:

STRONG YES indicators:
- Presence of AD/ADRD medications (Donepezil, Memantine, Tacrine, Rivastigmine, Galantamine)
- Explicit mention of Alzheimer's or dementia diagnosis in provider notes
- Any explicit diagnosis or strong provider impression of a related dementia subtype
  such as frontotemporal dementia, Lewy body dementia, vascular dementia,
  or major neurocognitive disorder should count as YES for AD/ADRD overall
- Cognitive test scores below normal threshold
- Brain imaging showing cortical atrophy or hippocampal volume loss

SUPPORTING YES indicators:
- Documented memory loss, confusion, or cognitive decline
- Long standing cognitive symptoms
- Prior MCI diagnosis
- Age > 65 with multiple cognitive symptoms
- Cardiovascular comorbidities alongside cognitive symptoms

NO indicators:
- No cognitive symptoms anywhere in chart
- No AD medications prescribed
- Normal cognitive test scores
- Confusion clearly explained by another condition such as delirium or infection
- No evidence of Alzheimer's disease or any related dementia disorder

UNCERTAIN indicators:
- Conflicting information in the chart
- Cognitive symptoms present but no formal diagnosis confirmed
- Insufficient information to decide confidently
- Symptoms potentially explained by another condition

IMPORTANT REMINDERS:
- UNCERTAIN is better than a wrong answer
- This task is AD/ADRD overall, not only Alzheimer's disease
- If the patient appears to have a different dementia subtype, classify as YES, not NO
- Distinguish "not classic Alzheimer's disease" from "no dementia-related disorder"
- Weigh strong indicators more heavily than supporting indicators
- Consider the full clinical picture, not just one signal in isolation
""".strip()

In [33]:
MODEL_NAME = "gpt-4o-mini"

In [34]:
def extract_one_note(row):
    prompt = build_extractor_prompt(row)

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            messages=[
                {"role": "system", "content": "Return only the required JSON object."},
                {"role": "user", "content": prompt}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": EXTRACTOR_SCHEMA
            }
        )

        content = response.choices[0].message.content
        parsed = json.loads(content)

        return {
            "extractor_status": "ok",
            "extracted_json": json.dumps(parsed, ensure_ascii=False),
            "extractor_raw_response": content
        }

    except Exception as e:
        return {
            "extractor_status": "api_error",
            "extracted_json": None,
            "extractor_raw_response": str(e)
        }

In [35]:
def confidence_to_prob(conf):
    mapping = {
        "HIGH": 0.90,
        "MEDIUM": 0.70,
        "LOW": 0.55
    }
    return mapping.get(conf, None)

def class_to_binary(label):
    if label == "YES":
        return 1
    if label == "NO":
        return 0
    return None

In [36]:
def classify_from_extraction(row, extracted_json_str):
    prompt = build_classifier_prompt(row, extracted_json_str)

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            messages=[
                {"role": "system", "content": "Return only the required JSON object."},
                {"role": "user", "content": prompt}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": CLASSIFIER_SCHEMA
            }
        )

        content = response.choices[0].message.content
        parsed = json.loads(content)

        classification = parsed["classification"]
        confidence = parsed["confidence"]

        return {
            "classifier_status": "ok",
            "classification_3way": classification,
            "confidence_level": confidence,
            "pred_label": class_to_binary(classification),
            "pred_prob": confidence_to_prob(confidence),
            "strong_indicators": json.dumps(parsed["strong_indicators"], ensure_ascii=False),
            "supporting_indicators": json.dumps(parsed["supporting_indicators"], ensure_ascii=False),
            "contradicting_indicators": json.dumps(parsed["contradicting_indicators"], ensure_ascii=False),
            "reasoning_summary": parsed["reasoning"],
            "conclusion": parsed["conclusion"],
            "classifier_raw_response": content
        }

    except Exception as e:
        return {
            "classifier_status": "api_error",
            "classification_3way": None,
            "confidence_level": None,
            "pred_label": None,
            "pred_prob": None,
            "strong_indicators": None,
            "supporting_indicators": None,
            "contradicting_indicators": None,
            "reasoning_summary": None,
            "conclusion": None,
            "classifier_raw_response": str(e)
        }

In [37]:
mini_test_df = test_df.head(2).copy()

mini_outputs = []

for _, row in mini_test_df.iterrows():
    row_dict = row.to_dict()

    ext_result = extract_one_note(row_dict)

    if ext_result["extractor_status"] == "ok":
        clf_result = classify_from_extraction(row_dict, ext_result["extracted_json"])
    else:
        clf_result = {
            "classifier_status": "skipped_due_to_extractor_error",
            "classification_3way": None,
            "confidence_level": None,
            "pred_label": None,
            "pred_prob": None,
            "strong_indicators": None,
            "supporting_indicators": None,
            "contradicting_indicators": None,
            "reasoning_summary": None,
            "conclusion": None,
            "classifier_raw_response": None
        }

    merged = {}
    merged.update(ext_result)
    merged.update(clf_result)
    mini_outputs.append(merged)

mini_pred_df = pd.concat(
    [mini_test_df.reset_index(drop=True), pd.DataFrame(mini_outputs)],
    axis=1
)

mini_pred_df[[
    "subject_id",
    "ground_truth",
    "extractor_status",
    "classifier_status",
    "classification_3way",
    "confidence_level",
    "pred_label",
    "pred_prob"
]]

,subject_id,ground_truth,extractor_status,classifier_status,classification_3way,confidence_level,pred_label,pred_prob
0,16983240,1,ok,ok,UNCERTAIN,MEDIUM,NaN,0.7
1,18513773,0,ok,ok,NO,HIGH,0.0,0.9


In [38]:
results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    row_dict = row.to_dict()

    ext_result = extract_one_note(row_dict)

    if ext_result["extractor_status"] == "ok":
        clf_result = classify_from_extraction(row_dict, ext_result["extracted_json"])
    else:
        clf_result = {
            "classifier_status": "skipped_due_to_extractor_error",
            "classification_3way": None,
            "confidence_level": None,
            "pred_label": None,
            "pred_prob": None,
            "strong_indicators": None,
            "supporting_indicators": None,
            "contradicting_indicators": None,
            "reasoning_summary": None,
            "conclusion": None,
            "classifier_raw_response": None
        }

    merged = {}
    merged.update(ext_result)
    merged.update(clf_result)
    results.append(merged)

pred_df = pd.concat(
    [test_df.reset_index(drop=True), pd.DataFrame(results)],
    axis=1
)

print(pred_df["extractor_status"].value_counts(dropna=False))
print(pred_df["classifier_status"].value_counts(dropna=False))
print(pred_df["classification_3way"].value_counts(dropna=False))

pred_df.to_csv("milestone2_results.csv", index=False)
print("\nResults saved to milestone2_results.csv!")

# Download to your computer
from google.colab import files
files.download("milestone2_results.csv")
print("Download started!")

  0%|          | 0/80 [00:00<?, ?it/s]

extractor_status
ok    80
Name: count, dtype: int64
classifier_status
ok    80
Name: count, dtype: int64
classification_3way
YES          50
NO           17
UNCERTAIN    13
Name: count, dtype: int64

Results saved to milestone2_results.csv!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!


In [40]:
eval_df = pred_df.copy()

eval_df = eval_df[
    eval_df["pred_label"].notna() &
    eval_df["pred_prob"].notna()
].copy()

eval_df["pred_label"] = eval_df["pred_label"].astype(int)
eval_df["pred_prob"] = eval_df["pred_prob"].astype(float)

print("Rows available for evaluation:", len(eval_df), "out of", len(pred_df))

Rows available for evaluation: 67 out of 80


In [39]:
import pandas as pd

# Load GPT-4o results
gpt_df = pd.read_csv("milestone2_results.csv")

# Basic checks
print("Shape:", gpt_df.shape)
print("\nColumns:", gpt_df.columns.tolist())

print("\nClassification distribution:")
print(gpt_df["classification_3way"].value_counts(dropna=False))

print("\nConfidence distribution:")
print(gpt_df["confidence_level"].value_counts(dropna=False))

print("\nExtractor status:")
print(gpt_df["extractor_status"].value_counts(dropna=False))

print("\nClassifier status:")
print(gpt_df["classifier_status"].value_counts(dropna=False))

print("\nMissing values:")
print(gpt_df.isnull().sum())

print("\nSample row:")
print(gpt_df.iloc[0])

Shape: (80, 34)

Columns: ['subject_id', 'hadm_id', 'gender', 'admit_age', 'race', 'text', 'radiology_notes', 'donepezil', 'memantine', 'tacrine', 'rivastigmine', 'galantamine', 'Enalapril', 'Furosemide', 'Lisinopril', 'Metoprolol', 'Spironolactone', 'ground_truth', 'combined_note', 'combined_note_trimmed', 'extractor_status', 'extracted_json', 'extractor_raw_response', 'classifier_status', 'classification_3way', 'confidence_level', 'pred_label', 'pred_prob', 'strong_indicators', 'supporting_indicators', 'contradicting_indicators', 'reasoning_summary', 'conclusion', 'classifier_raw_response']

Classification distribution:
classification_3way
YES          50
NO           17
UNCERTAIN    13
Name: count, dtype: int64

Confidence distribution:
confidence_level
HIGH      67
MEDIUM    13
Name: count, dtype: int64

Extractor status:
extractor_status
ok    80
Name: count, dtype: int64

Classifier status:
classifier_status
ok    80
Name: count, dtype: int64

Missing values:
subject_id          

In [45]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# ── Load results ──
pred_df = pd.read_csv("/content/milestone2_results.csv")  # replace with actual filename

# ══════════════════════════════════════
# STEP 1 — UNCERTAIN RATE (all 80 pts)
# ══════════════════════════════════════
total = len(pred_df)
uncertain_count = (pred_df["classification_3way"] == "UNCERTAIN").sum()
uncertain_rate  = uncertain_count / total * 100

print("=" * 50)
print("OVERALL DISTRIBUTION (80 patients)")
print("=" * 50)
print(f"YES:       {(pred_df['classification_3way'] == 'YES').sum()}")
print(f"NO:        {(pred_df['classification_3way'] == 'NO').sum()}")
print(f"UNCERTAIN: {uncertain_count} ({uncertain_rate:.1f}%)")

# ══════════════════════════════════════
# STEP 2 — EXCLUDE UNCERTAIN
# ══════════════════════════════════════
binary_df = pred_df[
    pred_df["classification_3way"] != "UNCERTAIN"
].copy()

print(f"\nPatients for binary metrics: {len(binary_df)}")
print(f"UNCERTAIN excluded:          {total - len(binary_df)}")

# ══════════════════════════════════════
# STEP 3 — BINARY PREDICTIONS
# ══════════════════════════════════════
binary_df["pred_binary"] = (
    binary_df["classification_3way"] == "YES"
).astype(int)

# ══════════════════════════════════════
# STEP 4 — ADJUSTED PROBABILITY FOR AUC
# ══════════════════════════════════════
def get_adjusted_prob(row):
    clf  = row["classification_3way"]
    conf = row["confidence_level"]
    if clf == "YES":
        if conf == "HIGH":   return 0.95
        elif conf == "MEDIUM": return 0.75
        else:                return 0.60
    elif clf == "NO":
        if conf == "HIGH":   return 0.05
        elif conf == "MEDIUM": return 0.25
        else:                return 0.40
    else:
        return 0.50

binary_df["adjusted_prob"] = binary_df.apply(
    get_adjusted_prob, axis=1
)

# ══════════════════════════════════════
# STEP 5 — CALCULATE ALL METRICS
# ══════════════════════════════════════
auc = roc_auc_score(
    binary_df["ground_truth"],
    binary_df["adjusted_prob"]
)
accuracy    = accuracy_score(
    binary_df["ground_truth"],
    binary_df["pred_binary"]
)
sensitivity = recall_score(
    binary_df["ground_truth"],
    binary_df["pred_binary"],
    zero_division=0
)
ppv         = precision_score(
    binary_df["ground_truth"],
    binary_df["pred_binary"],
    zero_division=0
)
f1          = f1_score(
    binary_df["ground_truth"],
    binary_df["pred_binary"],
    zero_division=0
)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    binary_df["ground_truth"],
    binary_df["pred_binary"]
).ravel()

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

# ══════════════════════════════════════
# STEP 6 — PRINT RESULTS
# ══════════════════════════════════════
print("\n" + "=" * 50)
print("GPT-4o FINAL METRICS (New Dataset)")
print("=" * 50)
print(f"AUC:               {auc:.4f}")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Sensitivity:       {sensitivity:.4f}")
print(f"Specificity:       {specificity:.4f}")
print(f"PPV (Precision):   {ppv:.4f}")
print(f"NPV:               {npv:.4f}")
print(f"F1 Score:          {f1:.4f}")
print(f"UNCERTAIN Rate:    {uncertain_rate:.1f}%")
print(f"\nConfusion Matrix:")
print(f"  TP: {tp}   FP: {fp}")
print(f"  FN: {fn}   TN: {tn}")
print(f"\nPatients evaluated: {len(binary_df)}/80")
print(f"UNCERTAIN excluded: {uncertain_count}")

# ══════════════════════════════════════
# STEP 7 — GROUND TRUTH DISTRIBUTION
# ══════════════════════════════════════
print("\n" + "=" * 50)
print("GROUND TRUTH CHECK")
print("=" * 50)
print(f"Positive (AD/ADRD): {(pred_df['ground_truth'] == 1).sum()}")
print(f"Negative:           {(pred_df['ground_truth'] == 0).sum()}")

# ══════════════════════════════════════
# STEP 8 — HALLUCINATION CHECK
# ══════════════════════════════════════
import json

hallucination_count   = 0
hallucination_details = []

ad_meds = [
    "donepezil", "memantine", "tacrine",
    "rivastigmine", "galantamine"
]

for _, row in pred_df.iterrows():
    try:
        extracted = json.loads(row["extracted_json"])
    except:
        continue

    # Get LLM reported AD medications
    llm_meds = []
    cog_meds = extracted.get("cognitive_medications", {})
    if isinstance(cog_meds, dict):
        llm_meds = cog_meds.get("ad_medications_present", [])
    elif isinstance(cog_meds, list):
        llm_meds = cog_meds

    for med in llm_meds:
        med_lower = str(med).lower()
        for actual_med in ad_meds:
            if actual_med in med_lower:
                actual_val = row.get(actual_med, 0)
                if actual_val == 0 or actual_val == "Not Filled":
                    hallucination_count += 1
                    hallucination_details.append({
                        "subject_id": row["subject_id"],
                        "hallucinated": med,
                        "actual_value": actual_val
                    })

print("\n" + "=" * 50)
print("HALLUCINATION CHECK")
print("=" * 50)
print(f"Total hallucinations: {hallucination_count}")
if hallucination_details:
    print("\nDetails:")
    for d in hallucination_details:
        print(f"  Patient {d['subject_id']}: {d['hallucinated']}")
else:
    print("No hallucinations detected")

# ══════════════════════════════════════
# STEP 9 — CONFIDENCE CALIBRATION
# ══════════════════════════════════════
print("\n" + "=" * 50)
print("CONFIDENCE CALIBRATION")
print("=" * 50)

for conf in ["HIGH", "MEDIUM"]:
    conf_df = binary_df[binary_df["confidence_level"] == conf]
    if len(conf_df) == 0:
        continue
    correct = (conf_df["pred_binary"] == conf_df["ground_truth"]).sum()
    acc = correct / len(conf_df) * 100
    print(f"{conf} confidence: {len(conf_df)} patients → {acc:.1f}% correct")

# ══════════════════════════════════════
# STEP 10 — SAVE METRICS SUMMARY
# ══════════════════════════════════════
metrics_summary = pd.DataFrame({
    "Metric": [
        "AUC", "Accuracy", "Sensitivity",
        "Specificity", "PPV", "NPV",
        "F1 Score", "UNCERTAIN Rate",
        "TP", "FP", "FN", "TN",
        "Hallucinations",
        "Patients Evaluated"
    ],
    "GPT-4o (New Dataset)": [
        round(auc, 4),
        round(accuracy, 4),
        round(sensitivity, 4),
        round(specificity, 4),
        round(ppv, 4),
        round(npv, 4),
        round(f1, 4),
        f"{uncertain_rate:.1f}%",
        tp, fp, fn, tn,
        hallucination_count,
        len(binary_df)
    ]
})

metrics_summary.to_csv(
    "gpt4o_metrics_ML2.csv",
    index=False
)

from google.colab import files
files.download("gpt4o_metrics_ML2.csv")
print("\n✅ Metrics saved and downloaded!")

OVERALL DISTRIBUTION (80 patients)
YES:       50
NO:        17
UNCERTAIN: 13 (16.2%)

Patients for binary metrics: 67
UNCERTAIN excluded:          13

GPT-4o FINAL METRICS (New Dataset)
AUC:               0.9378
Accuracy:          0.9403
Sensitivity:       0.9423
Specificity:       0.9333
PPV (Precision):   0.9800
NPV:               0.8235
F1 Score:          0.9608
UNCERTAIN Rate:    16.2%

Confusion Matrix:
  TP: 49   FP: 1
  FN: 3   TN: 14

Patients evaluated: 67/80
UNCERTAIN excluded: 13

GROUND TRUTH CHECK
Positive (AD/ADRD): 60
Negative:           20

HALLUCINATION CHECK
Total hallucinations: 13

Details:
  Patient 16540501: Donepezil
  Patient 16611643: Donepezil 5 mg Tablet Sig: Two (2) Tablet PO HS (at bedtime)
  Patient 16881510: DONEPEZIL [ARICEPT] - 10 mg Tablet - 1 Tablet(s) by mouth at bedtime
  Patient 15776779: Donepezil
  Patient 15776779: Memantine
  Patient 12973666: Donepezil
  Patient 18620666: Donepezil 10 mg PO QHS
  Patient 14418295: Donepezil
  Patient 11527327:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Metrics saved and downloaded!


In [46]:
import json

# Check if hallucinated meds appear in text
suspicious_ids = [
    16540501, 16611643, 16881510,
    15776779, 12973666, 18620666,
    14418295, 11527327, 17616643,
    13318285, 17240084
]

print("Checking if medications appear in text...")
print("=" * 60)

for pid in suspicious_ids:
    patient = pred_df[pred_df["subject_id"] == pid]
    if len(patient) == 0:
        continue

    row = patient.iloc[0]
    text = str(row["text"]).lower()
    rad  = str(row["radiology_notes"]).lower()

    donepezil_in_text = "donepezil" in text or "aricept" in text
    memantine_in_text = "memantine" in text or "namenda" in text

    don_col = row["donepezil"]
    mem_col = row["memantine"]

    print(f"\nPatient {pid}:")
    print(f"  donepezil col={don_col} | in_text={donepezil_in_text}")
    print(f"  memantine col={mem_col} | in_text={memantine_in_text}")

    if donepezil_in_text and don_col == 0:
        print(f"  ⚠️  Donepezil in TEXT but col=0 → Column missing data")
    elif not donepezil_in_text and don_col == 0:
        print(f"  ❌  Donepezil NOT in text and col=0 → True hallucination")

Checking if medications appear in text...

Patient 16540501:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=False
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 16611643:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=False
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 16881510:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=False
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 15776779:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=True
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 12973666:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=False
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 18620666:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text=False
  ⚠️  Donepezil in TEXT but col=0 → Column missing data

Patient 14418295:
  donepezil col=0.0 | in_text=True
  memantine col=0.0 | in_text

In [47]:
# Revised hallucination analysis
print("=" * 50)
print("REVISED HALLUCINATION ANALYSIS")
print("=" * 50)
print(f"Apparent hallucinations:      13")
print(f"True hallucinations:           0")
print(f"Data quality issues:          13")
print(f"\nAll 13 cases: medication present")
print(f"in discharge text but col = 0")
print(f"GPT-4o correctly extracted from text")
print(f"Structured column was incomplete")

# Check how many patients this affected
affected = pred_df[
    pred_df["subject_id"].isin(suspicious_ids)
]["classification_3way"].value_counts()

print(f"\nClassification for affected patients:")
print(affected)

# Were they correctly classified?
affected_df = pred_df[
    pred_df["subject_id"].isin(suspicious_ids)
].copy()

correct = (
    ((affected_df["classification_3way"] == "YES") &
     (affected_df["ground_truth"] == 1)) |
    ((affected_df["classification_3way"] == "NO") &
     (affected_df["ground_truth"] == 0))
).sum()

print(f"\nOf 13 affected patients:")
print(f"Correctly classified: {correct}")
print(f"Incorrectly classified: {13 - correct}")

REVISED HALLUCINATION ANALYSIS
Apparent hallucinations:      13
True hallucinations:           0
Data quality issues:          13

All 13 cases: medication present
in discharge text but col = 0
GPT-4o correctly extracted from text
Structured column was incomplete

Classification for affected patients:
classification_3way
YES    11
Name: count, dtype: int64

Of 13 affected patients:
Correctly classified: 11
Incorrectly classified: 2
